# College Track Data: Athletes, Schools, and Conferences
I compare women's outdoor **400m, 100H, 400H, and long jump** results from **2016–2026**.

For each event, the charts appear in this order: **athletes → schools → conferences**. I include every result that can be read as a positive time or distance, not just season bests. All rows remain in the data summary.

**How to run:** Choose **Kernel → Restart Kernel and Run All Cells**. The notebook reads my event files directly from GitHub, so I need an internet connection. Charts save in `outputs/track_comparisons`. Generating every athlete's charts can take several minutes.

**How to read the charts:** Years go across the bottom. Each dot is one recorded result. Lower times are faster; higher jump distances are farther. I use the same vertical scale within each event to make comparisons easier. Dots are spread slightly around their year so matching results are easier to see; this spacing does not show meet dates.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA = 'https://raw.githubusercontent.com/jaiamcgee/collegetrackperformances/main/data/processed/by_event'
OUT = Path.cwd() / 'outputs' / 'track_comparisons'
OUT.mkdir(parents=True, exist_ok=True)
FILES = {'400m':'400m_2016_2026.csv', '100H':'100H_2016_2026.csv',
         '400H':'400H_2016_2026.csv', 'Long Jump':'long_jump_2016_2026.csv'}
YEARS = list(range(2016, 2027))
GROUPS = ['ACC', 'SEC', 'Big Ten', 'Big 12']
COLORS = dict(zip(GROUPS, ['#6639ba','#dc3057','#007e99','#cc7900']))
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi':120, 'font.size':12, 'axes.titlesize':14, 'axes.labelsize':12, 'axes.spines.top':False, 'axes.spines.right':False})
frames = {}
for event, filename in FILES.items():
    df = pd.read_csv(f"{DATA}/{filename}", low_memory=False)
    required = {'athlete','school','conference','season','performance','unit','wind','data_status'}
    if required - set(df.columns):
        raise ValueError(f'{filename}: missing columns {required - set(df.columns)}')
    for col in ['athlete','school','conference','unit','data_status']:
        df[col] = df[col].astype('string').str.strip().replace('', pd.NA)
    df['season'] = pd.to_numeric(df['season'], errors='coerce')
    df['event_file'] = event
    frames[event] = df
all_data = pd.concat(frames.values(), ignore_index=True)
print(f'Loaded {len(all_data):,} recorded rows. Output folder: {OUT}')
display(pd.DataFrame({'event':list(frames), 'rows':[len(d) for d in frames.values()]}))

def finish(fig, name):
    fig.tight_layout()
    fig.savefig(OUT / f'{name}.png', dpi=180, bbox_inches='tight')
    plt.show()

## Check the data first
I keep every source row. A blank year means I have no rows for that year in this file; athletes may still have competed. Having results does not mean I collected the whole season.

Some results still need source review. Wind-assisted and missing-wind results remain included. Conference names describe my project groups, which may differ from the schools' conferences in earlier years.

In [ ]:
def parse_mark(row):
    text = str(row['performance']).strip()
    unit = str(row['unit']).lower()
    if row['event_file'] == 'Long Jump':
        if unit not in {'m','meters','metres'}:
            return np.nan
        return float(text) if re.fullmatch(r'\d+(?:\.\d+)?',text) else np.nan
    if unit not in {'s','seconds','sec'}:
        return np.nan
    if re.fullmatch(r'\d+(?:\.\d+)?',text):
        return float(text)
    if re.fullmatch(r'\d+:[0-5]\d(?:\.\d+)?',text):
        minutes,seconds=text.split(':')
        return 60*float(minutes)+float(seconds)
    return np.nan

analysis = all_data.copy()
analysis['mark'] = analysis.apply(parse_mark, axis=1)
analysis['wind_numeric'] = pd.to_numeric(analysis['wind'], errors='coerce')
analysis['numeric_result'] = analysis['mark'].gt(0)
analysis.to_csv(OUT / 'all_records_with_numeric_marks.csv', index=False)
quality = analysis.groupby('event_file').agg(recorded_rows=('mark','size'), plottable_results=('numeric_result','sum')).reindex(FILES)
quality['non_numeric_or_non_positive'] = quality['recorded_rows'] - quality['plottable_results']
display(quality)
quality.to_csv(OUT / 'numeric_result_summary.csv')
analysis.loc[~analysis['numeric_result']].to_csv(OUT / 'unplottable_results_to_review.csv',index=False)
print('No wind, review-status, athlete, or season-best filters applied.')

## Chart settings
The notebook creates every comparison automatically. Athlete charts show six athlete-school combinations per page. School and conference charts show four groups per page. An athlete recorded at different schools appears in separate panels.

Colors identify the project conference groups. The dots show actual results, not averages. Athletes with more recorded performances contribute more dots.

In [ ]:
plt.rcParams.update({'font.size':12, 'axes.titlesize':14, 'axes.labelsize':12})

def compare_through_years(event, column, title, per_page):
    d = analysis.loc[analysis['event_file'].eq(event)
        & analysis['numeric_result'] & analysis['season'].isin(YEARS)].copy()
    if column == 'athlete_school':
        d[column] = d['athlete'].fillna('Unknown') + ' — ' + d['school'].fillna('Unknown')
    else:
        d[column] = d[column].fillna('Unknown')
    labels = sorted(d[column].unique())
    if not labels:
        print(f'{event}: no numeric results available.')
        return
    low, high = d['mark'].min(), d['mark'].max()
    margin = max((high-low)*.08, .15)
    pages = (len(labels)+per_page-1)//per_page
    print(f'{event}: {len(labels)} {title.lower()} across {pages} chart pages.')
    rng = np.random.default_rng(42)
    d['display_year'] = d['season'] + rng.uniform(-.17,.17,len(d))
    for page in range(pages):
        names = labels[page*per_page:(page+1)*per_page]
        rows = (per_page+1)//2
        fig, axes = plt.subplots(rows,2,figsize=(17,rows*3.8),sharex=True,sharey=True)
        for ax,name in zip(axes.flat,names):
            g = d.loc[d[column].eq(name)]
            for conference,cg in g.groupby(g['conference'].fillna('Unknown')):
                ax.scatter(cg['display_year'],cg['mark'],s=42,alpha=.72,
                    color=COLORS.get(conference,'#555555'),edgecolors='white',linewidths=.35)
            ax.set_title(f'{name}\n{len(g):,} recorded results',fontsize=12,fontweight='bold')
            ax.set_xticks(YEARS)
            ax.set_xlim(2015.5,2026.5)
            ax.set_ylim(low-margin,high+margin)
            ax.tick_params(axis='x',labelbottom=True,rotation=45,labelsize=10)
            ax.set_xlabel('Year')
            ax.set_ylabel('Jump distance (m)' if event=='Long Jump' else 'Race time (seconds)')
            ax.set_facecolor('#f7f8fc')
            ax.grid(True,color='#dce0e8',linewidth=.6)
        for ax in list(axes.flat)[len(names):]: ax.set_visible(False)
        fig.suptitle(f'{event} • {title} through the years • Page {page+1} of {pages}',
                     fontsize=20,fontweight='bold',y=1.01)
        from matplotlib.lines import Line2D
        handles=[Line2D([0],[0],marker='o',color='none',markerfacecolor=COLORS[g],markersize=8,label=g)
                 for g in GROUPS if d['conference'].eq(g).any()]
        fig.legend(handles=handles,loc='upper center',bbox_to_anchor=(.5,.985),ncol=4,frameon=False)
        fig.tight_layout(rect=[0,0,1,.945])
        filename = f"{event.replace(' ','_')}_{column}_page_{page+1:03d}"
        fig.savefig(OUT / f'{filename}.png',dpi=150,bbox_inches='tight')
        plt.show()
    print('Saved every page to:',OUT)

# 400m
I show every numeric result for this event from 2016–2026. Lower dots mean faster race times. Missing years remain visible as gaps.

## 400m: athletes
Each panel shows one athlete at one school. I compare their results across all recorded years, using the same scale for every athlete in this event.

In [ ]:
compare_through_years('400m', 'athlete_school', 'Athletes', per_page=6)

## 400m: schools
Each panel shows one school. Every dot is a recorded result from an athlete at that school.

In [ ]:
compare_through_years('400m', 'school', 'Schools', per_page=4)

## 400m: conferences
Each panel shows one project conference group. Every dot is a recorded result from one of its selected schools.

In [ ]:
compare_through_years('400m', 'conference', 'Conferences', per_page=4)

# 100H
I show every numeric result for this event from 2016–2026. Lower dots mean faster race times. Missing years remain visible as gaps.

## 100H: athletes
Each panel shows one athlete at one school. I compare their results across all recorded years, using the same scale for every athlete in this event.

In [ ]:
compare_through_years('100H', 'athlete_school', 'Athletes', per_page=6)

## 100H: schools
Each panel shows one school. Every dot is a recorded result from an athlete at that school.

In [ ]:
compare_through_years('100H', 'school', 'Schools', per_page=4)

## 100H: conferences
Each panel shows one project conference group. Every dot is a recorded result from one of its selected schools.

In [ ]:
compare_through_years('100H', 'conference', 'Conferences', per_page=4)

# 400H
I show every numeric result for this event from 2016–2026. Lower dots mean faster race times. Missing years remain visible as gaps.

## 400H: athletes
Each panel shows one athlete at one school. I compare their results across all recorded years, using the same scale for every athlete in this event.

In [ ]:
compare_through_years('400H', 'athlete_school', 'Athletes', per_page=6)

## 400H: schools
Each panel shows one school. Every dot is a recorded result from an athlete at that school.

In [ ]:
compare_through_years('400H', 'school', 'Schools', per_page=4)

## 400H: conferences
Each panel shows one project conference group. Every dot is a recorded result from one of its selected schools.

In [ ]:
compare_through_years('400H', 'conference', 'Conferences', per_page=4)

# Long Jump
I show every numeric result for this event from 2016–2026. Higher dots mean longer jumps. Missing years remain visible as gaps.

## Long Jump: athletes
Each panel shows one athlete at one school. I compare their results across all recorded years, using the same scale for every athlete in this event.

In [ ]:
compare_through_years('Long Jump', 'athlete_school', 'Athletes', per_page=6)

## Long Jump: schools
Each panel shows one school. Every dot is a recorded result from an athlete at that school.

In [ ]:
compare_through_years('Long Jump', 'school', 'Schools', per_page=4)

## Long Jump: conferences
Each panel shows one project conference group. Every dot is a recorded result from one of its selected schools.

In [ ]:
compare_through_years('Long Jump', 'conference', 'Conferences', per_page=4)

## What I can learn from these charts
I can compare how results vary across athletes, schools, conference groups, and years. I also need to check missing records and source quality before explaining why differences exist. The charts do not show complete conference participation or prove that an athlete improved.

I can write my observations here after reviewing the charts. I save the notebook with **Command + S** so GitHub can display the chart outputs.